# CIFAR-10 Image Classification -- Further experiments  
  
- cifar10_experiments.ipynb  
- Phase 4 -- Full Dataset Experiments  
- S10: Head Only, 50k training samples  

## S1 - Imports & Configuration

In [30]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix
from dotenv import load_dotenv

load_dotenv()
PROJECT_PATH = os.getenv('PROJECT_ROOT')
DATA_PATH = os.getenv('DATA_PATH')
CHECKPOINT_DIR = os.path.join(PROJECT_PATH, 'checkpoints')

torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


## S2 - Data Loading & Sampling

In [24]:
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Transforms -- identical to Phase 3 (controlled comparison)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Full 50k training set -- no Subset cap
full_train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_PATH, train=True, download=True, transform=train_transform
)

# 80/20 split --> ~40k train, ~10k val
val_size = int(0.2 * len(full_train_dataset))
train_size = len(full_train_dataset) - val_size
train_split, val_split = random_split(full_train_dataset, [train_size, val_size])

# Increase num_workers to utilize your CPU cores
# Try 6-8 on a robust CPU with 32GB RAM
train_loader = DataLoader(train_split, batch_size=32, shuffle=True,
                          num_workers=6, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_split,   batch_size=32, shuffle=False,
                          num_workers=6, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False,
                          num_workers=6, pin_memory=True, persistent_workers=True)
# train_loader = DataLoader(train_split, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
# val_loader   = DataLoader(val_split,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_PATH, train=False, download=True, transform=test_transform
)
# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_split)} | Val: {len(val_split)} | Test: {len(test_dataset)}")

Files already downloaded and verified
Files already downloaded and verified
Train: 40000 | Val: 10000 | Test: 10000


## S3 - Model Setup

In [25]:
# S10: Model Setup -- Fresh ResNet50, Frozen Base, Same Head as Phase 3

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Load pretrained ResNet50 -- fresh ImageNet weights (not from S7 checkpoint)
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze all base layers
for param in resnet50.parameters():
    param.requires_grad = False

# Same custom head as Phase 3
resnet50.fc = nn.Sequential(
    nn.Flatten(),
    nn.Linear(2048, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 10)
)

resnet50 = resnet50.to(device)

trainable = sum(p.numel() for p in resnet50.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")
# Expected: 271,178 (head only)

# Optimizer + Loss
optimizer_s10 = optim.Adam(
    filter(lambda p: p.requires_grad, resnet50.parameters()),
    lr=1e-3
)
criterion = nn.CrossEntropyLoss()

NUM_EPOCHS = 10
PATIENCE   = 3

print("S10 ready -- frozen base, Adam lr=1e-3, full 50k dataset")

Trainable parameters: 271,178
S10 ready -- frozen base, Adam lr=1e-3, full 50k dataset


## S4 Training Loops

##### Original FULL Dataset Head Only

In [7]:
# S10: Training Loop -- Head Only, Full Dataset

history_s10 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

best_val_loss    = float('inf')
patience_counter = 0
best_model_state_s10 = None

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    resnet50.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_s10.zero_grad()
        outputs = resnet50(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_s10.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # --- Validate ---
    resnet50.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet50(images)
            loss = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total   += labels.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total

    history_s10['train_loss'].append(train_loss)
    history_s10['train_acc'].append(train_acc)
    history_s10['val_loss'].append(val_loss)
    history_s10['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"Patience: {patience_counter}/{PATIENCE}")

    # --- EarlyStopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state_s10 = resnet50.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

# Save checkpoint
resnet50.load_state_dict(best_model_state_s10)
S10_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 's10_best_weights.pth')
torch.save(best_model_state_s10, S10_CHECKPOINT)
print(f"S10 best weights saved: {S10_CHECKPOINT}")

Epoch 01 | Train Loss: 0.9210 | Train Acc: 0.6811 | Val Loss: 0.6006 | Val Acc: 0.7941 | Patience: 0/3
Epoch 02 | Train Loss: 0.7049 | Train Acc: 0.7606 | Val Loss: 0.5742 | Val Acc: 0.7991 | Patience: 0/3
Epoch 03 | Train Loss: 0.6520 | Train Acc: 0.7774 | Val Loss: 0.5867 | Val Acc: 0.7926 | Patience: 0/3
Epoch 04 | Train Loss: 0.6291 | Train Acc: 0.7879 | Val Loss: 0.5740 | Val Acc: 0.7958 | Patience: 1/3
Epoch 05 | Train Loss: 0.6141 | Train Acc: 0.7929 | Val Loss: 0.5523 | Val Acc: 0.8088 | Patience: 0/3
Epoch 06 | Train Loss: 0.5960 | Train Acc: 0.7975 | Val Loss: 0.5213 | Val Acc: 0.8140 | Patience: 0/3
Epoch 07 | Train Loss: 0.5852 | Train Acc: 0.8014 | Val Loss: 0.5480 | Val Acc: 0.8096 | Patience: 0/3
Epoch 08 | Train Loss: 0.5706 | Train Acc: 0.8081 | Val Loss: 0.5210 | Val Acc: 0.8165 | Patience: 1/3
Epoch 09 | Train Loss: 0.5593 | Train Acc: 0.8101 | Val Loss: 0.5477 | Val Acc: 0.8104 | Patience: 0/3
Epoch 10 | Train Loss: 0.5397 | Train Acc: 0.8184 | Val Loss: 0.5233 | Va

## Test Set Evaluations

In [26]:
# S10: Test Set Evaluation

import numpy as np
from sklearn.metrics import classification_report

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

def evaluate_model(model, loader, criterion, device, label):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / total
    accuracy = correct / total
    print(f"{label} | Test Loss: {avg_loss:.4f} | Test Acc: {accuracy:.4f} ({accuracy*100:.2f}%)")
    return avg_loss, accuracy, all_preds, all_labels

# Load S10 best weights and evaluate
resnet50.load_state_dict(torch.load(S10_CHECKPOINT, weights_only=True))
s10_loss, s10_acc, s10_preds, s10_labels = evaluate_model(
    resnet50, test_loader, criterion, device, 'S10 -- Head Only (50k)'
)

# Per-class accuracy
preds_np  = np.array(s10_preds)
labels_np = np.array(s10_labels)

print("\nS10 Per-Class Accuracy:")
for i, name in enumerate(CLASS_NAMES):
    mask = labels_np == i
    class_acc = (preds_np[mask] == i).mean()
    print(f"  {name:<12}: {class_acc:.2%}")

S10 -- Head Only (50k) | Test Loss: 0.5255 | Test Acc: 0.8178 (81.78%)

S10 Per-Class Accuracy:
  airplane    : 90.10%
  automobile  : 83.70%
  bird        : 83.00%
  cat         : 69.70%
  deer        : 75.30%
  dog         : 74.70%
  frog        : 82.30%
  horse       : 85.00%
  ship        : 83.20%
  truck       : 90.80%


### Ver 2 Head Only - 35k subset

## Segmentation Learning strategy

### Setup 35k Dataset

In [27]:
# S11: DataLoader -- 35k training subset
# Controlled comparison: layer3-only vs layer4-only, same data, same LR

from torch.utils.data import Subset

# Use first 35k samples from full training set
train_35k = Subset(full_train_dataset, range(35000))

# 80/20 split --> 28k train, 7k val
val_size_s11   = int(0.2 * len(train_35k))
train_size_s11 = len(train_35k) - val_size_s11

train_split_s11, val_split_s11 = random_split(train_35k, [train_size_s11, val_size_s11])

train_loader_s11 = DataLoader(train_split_s11, batch_size=32, shuffle=True,
                               num_workers=6, pin_memory=True, persistent_workers=True)
val_loader_s11   = DataLoader(val_split_s11,   batch_size=32, shuffle=False,
                               num_workers=6, pin_memory=True, persistent_workers=True)

print(f"S11 Train: {len(train_split_s11)} | Val: {len(val_split_s11)}")
# Expected: Train: 28000 | Val: 7000

S11 Train: 28000 | Val: 7000


### Layer 3 Unfreeze Only config

In [28]:
# S11-R1: Layer3 Unfreeze Only -- AdamW lr=1e-4
# Hypothesis: mid-level texture/edge filters in layer3 may adapt better
# to 32->224px upscaling artifacts than high-level layer4 features

# Fresh ResNet50 from ImageNet weights
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze all layers first
for param in resnet50.parameters():
    param.requires_grad = False

# Unfreeze layer3 only
for param in resnet50.layer3.parameters():
    param.requires_grad = True

# Same custom head as Phase 3
resnet50.fc = nn.Sequential(
    nn.Flatten(),
    nn.Linear(2048, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 10)
)

resnet50 = resnet50.to(device)

trainable = sum(p.numel() for p in resnet50.parameters() if p.requires_grad)
print(f"Trainable parameters (S11-R1): {trainable:,}")
# layer3 (~14.9M) + head (271k) -- expect ~15.2M

optimizer_s11r1 = optim.AdamW(
    filter(lambda p: p.requires_grad, resnet50.parameters()),
    lr=1e-4
)

print("S11-R1 ready -- layer3 unfrozen, AdamW lr=1e-4, 35k dataset")

Trainable parameters (S11-R1): 7,369,546
S11-R1 ready -- layer3 unfrozen, AdamW lr=1e-4, 35k dataset


### Layer 3 Only - Training Loop

##### Archived Training Loop  

This code created several gaps in:  
- saving the best model weights  
- patience cycle parameters  
- val_loss < best_val_loss
- min_delta settings

In [12]:
# S11-R1: Training Loop -- Layer3 Only, 35k dataset

history_s11r1 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

best_val_loss      = float('inf')
patience_counter   = 0
best_model_state_s11r1 = None

NUM_EPOCHS = 10
PATIENCE   = 3

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    resnet50.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader_s11:
        images, labels = images.to(device), labels.to(device)
        optimizer_s11r1.zero_grad()
        outputs = resnet50(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_s11r1.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # --- Validate ---
    resnet50.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader_s11:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet50(images)
            loss = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total   += labels.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total

    history_s11r1['train_loss'].append(train_loss)
    history_s11r1['train_acc'].append(train_acc)
    history_s11r1['val_loss'].append(val_loss)
    history_s11r1['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"Patience: {patience_counter}/{PATIENCE}")

    # --- EarlyStopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state_s11r1 = resnet50.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

# Save checkpoint
resnet50.load_state_dict(best_model_state_s11r1)
S11R1_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 's11r1_best_weights.pth')
torch.save(best_model_state_s11r1, S11R1_CHECKPOINT)
print(f"S11-R1 best weights saved: {S11R1_CHECKPOINT}")

Epoch 01 | Train Loss: 0.7691 | Train Acc: 0.7692 | Val Loss: 0.2849 | Val Acc: 0.9090 | Patience: 0/3
Epoch 02 | Train Loss: 0.2364 | Train Acc: 0.9279 | Val Loss: 0.2386 | Val Acc: 0.9219 | Patience: 0/3
Epoch 03 | Train Loss: 0.1303 | Train Acc: 0.9607 | Val Loss: 0.2307 | Val Acc: 0.9313 | Patience: 0/3
Epoch 04 | Train Loss: 0.0925 | Train Acc: 0.9713 | Val Loss: 0.2297 | Val Acc: 0.9293 | Patience: 0/3
Epoch 05 | Train Loss: 0.0694 | Train Acc: 0.9784 | Val Loss: 0.2501 | Val Acc: 0.9269 | Patience: 0/3
Epoch 06 | Train Loss: 0.0631 | Train Acc: 0.9806 | Val Loss: 0.2620 | Val Acc: 0.9253 | Patience: 1/3
Epoch 07 | Train Loss: 0.0560 | Train Acc: 0.9819 | Val Loss: 0.2871 | Val Acc: 0.9237 | Patience: 2/3
Early stopping triggered.
S11-R1 best weights saved: Q:\scripts\projects\ComputerVision-Term9\checkpoints\s11r1_best_weights.pth


In [13]:
# Verify checkpoint is epoch 3 weights
resnet50.load_state_dict(torch.load(S11R1_CHECKPOINT, weights_only=True))
loss, acc, _, _ = evaluate_model(resnet50, val_loader_s11, criterion, device, 'S11-R1 verify')
print(f"Expected val loss ~0.2307 | Got: {loss:.4f}")

S11-R1 verify | Test Loss: 0.2871 | Test Acc: 0.9237 (92.37%)
Expected val loss ~0.2307 | Got: 0.2871


#### Layer 3 Training Loop - corrected

In [29]:
# S11-R1: Training Loop (UPDATED) -- Layer3 Only, 35k dataset
# Fixes applied:
#   1. copy.deepcopy for correct best weight capture
#   2. min_delta threshold to prevent trivial patience resets

import copy

history_s11r1 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

best_val_loss        = float('inf')
patience_counter     = 0
best_model_state_s11r1 = None

NUM_EPOCHS = 10
PATIENCE   = 3
MIN_DELTA  = 0.005

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    resnet50.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader_s11:
        images, labels = images.to(device), labels.to(device)
        optimizer_s11r1.zero_grad()
        outputs = resnet50(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_s11r1.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # --- Validate ---
    resnet50.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader_s11:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet50(images)
            loss = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total   += labels.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total

    history_s11r1['train_loss'].append(train_loss)
    history_s11r1['train_acc'].append(train_acc)
    history_s11r1['val_loss'].append(val_loss)
    history_s11r1['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"Patience: {patience_counter}/{PATIENCE}")

    # --- EarlyStopping with min_delta ---
    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state_s11r1 = copy.deepcopy(resnet50.state_dict())
        print(f"  >> New best: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

# Save checkpoint
resnet50.load_state_dict(best_model_state_s11r1)
S11R1_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 's11r1_best_weights.pth')
torch.save(best_model_state_s11r1, S11R1_CHECKPOINT)
print(f"S11-R1 best weights saved: {S11R1_CHECKPOINT}")

Epoch 01 | Train Loss: 0.7449 | Train Acc: 0.7716 | Val Loss: 0.2845 | Val Acc: 0.9077 | Patience: 0/3
  >> New best: 0.2845
Epoch 02 | Train Loss: 0.2404 | Train Acc: 0.9240 | Val Loss: 0.1957 | Val Acc: 0.9340 | Patience: 0/3
  >> New best: 0.1957
Epoch 03 | Train Loss: 0.1270 | Train Acc: 0.9607 | Val Loss: 0.2222 | Val Acc: 0.9304 | Patience: 0/3
Epoch 04 | Train Loss: 0.0961 | Train Acc: 0.9706 | Val Loss: 0.2235 | Val Acc: 0.9299 | Patience: 1/3
Epoch 05 | Train Loss: 0.0730 | Train Acc: 0.9771 | Val Loss: 0.2318 | Val Acc: 0.9303 | Patience: 2/3
Early stopping triggered.
S11-R1 best weights saved: Q:\scripts\projects\ComputerVision-Term9\checkpoints\s11r1_best_weights.pth


#### Best Epoch weights saved - Verification

In [31]:
resnet50.load_state_dict(torch.load(S11R1_CHECKPOINT, weights_only=True))
loss, acc, _, _ = evaluate_model(resnet50, val_loader_s11, criterion, device, 'S11-R1 verify')
print(f"Expected val loss ~0.1957 | Got: {loss:.4f}")

S11-R1 verify | Test Loss: 0.1957 | Test Acc: 0.9340 (93.40%)
Expected val loss ~0.1957 | Got: 0.1957


#### Test Set Evaluations Compare

In [32]:
resnet50.load_state_dict(torch.load(S11R1_CHECKPOINT, weights_only=True))
s11r1_loss, s11r1_acc, s11r1_preds, s11r1_labels = evaluate_model(
    resnet50, test_loader, criterion, device, 'S11-R1 -- Layer3 Only (35k)'
)

preds_np  = np.array(s11r1_preds)
labels_np = np.array(s11r1_labels)

print("\nS11-R1 Per-Class Accuracy:")
for i, name in enumerate(CLASS_NAMES):
    mask = labels_np == i
    class_acc = (preds_np[mask] == i).mean()
    print(f"  {name:<12}: {class_acc:.2%}")

S11-R1 -- Layer3 Only (35k) | Test Loss: 0.2127 | Test Acc: 0.9287 (92.87%)

S11-R1 Per-Class Accuracy:
  airplane    : 95.20%
  automobile  : 95.20%
  bird        : 89.80%
  cat         : 83.40%
  deer        : 92.20%
  dog         : 90.10%
  frog        : 96.70%
  horse       : 94.50%
  ship        : 96.10%
  truck       : 95.50%


### Layer 4 Only - Training Loop

In [33]:
# S11-R2: Model Setup -- Layer4 Only, 35k dataset
# Controlled comparison against S11-R1 (layer3 only, same data, same LR)
# Hypothesis: layer4 (high-level features) will underperform layer3
# on low-res upscaled inputs due to artifact interference at mid-level frequencies

import copy
torch.backends.cudnn.benchmark = True

# Fresh ResNet50 from ImageNet weights
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze all layers first
for param in resnet50.parameters():
    param.requires_grad = False

# Unfreeze layer4 only
for param in resnet50.layer4.parameters():
    param.requires_grad = True

# Same custom head
resnet50.fc = nn.Sequential(
    nn.Flatten(),
    nn.Linear(2048, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 10)
)

resnet50 = resnet50.to(device)

trainable = sum(p.numel() for p in resnet50.parameters() if p.requires_grad)
print(f"Trainable parameters (S11-R2): {trainable:,}")
# Expected: ~8.4M (layer4) + 271k (head)

optimizer_s11r2 = optim.AdamW(
    filter(lambda p: p.requires_grad, resnet50.parameters()),
    lr=1e-4
)

print("S11-R2 ready -- layer4 unfrozen, AdamW lr=1e-4, 35k dataset")

# ── Training Loop ──────────────────────────────────────────────────────────────

history_s11r2 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

best_val_loss          = float('inf')
patience_counter       = 0
best_model_state_s11r2 = None

NUM_EPOCHS = 10
PATIENCE   = 3
MIN_DELTA  = 0.005

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    resnet50.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader_s11:
        images, labels = images.to(device), labels.to(device)
        optimizer_s11r2.zero_grad()
        outputs = resnet50(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_s11r2.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # --- Validate ---
    resnet50.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader_s11:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet50(images)
            loss = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total   += labels.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total

    history_s11r2['train_loss'].append(train_loss)
    history_s11r2['train_acc'].append(train_acc)
    history_s11r2['val_loss'].append(val_loss)
    history_s11r2['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"Patience: {patience_counter}/{PATIENCE}")

    # --- EarlyStopping with min_delta ---
    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state_s11r2 = copy.deepcopy(resnet50.state_dict())
        print(f"  >> New best: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

# Save checkpoint
resnet50.load_state_dict(best_model_state_s11r2)
S11R2_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 's11r2_best_weights.pth')
torch.save(best_model_state_s11r2, S11R2_CHECKPOINT)
print(f"S11-R2 best weights saved: {S11R2_CHECKPOINT}")

Trainable parameters (S11-R2): 15,235,914
S11-R2 ready -- layer4 unfrozen, AdamW lr=1e-4, 35k dataset
Epoch 01 | Train Loss: 0.6671 | Train Acc: 0.7959 | Val Loss: 0.3277 | Val Acc: 0.8926 | Patience: 0/3
  >> New best: 0.3277
Epoch 02 | Train Loss: 0.2762 | Train Acc: 0.9130 | Val Loss: 0.3118 | Val Acc: 0.9007 | Patience: 0/3
  >> New best: 0.3118
Epoch 03 | Train Loss: 0.1532 | Train Acc: 0.9524 | Val Loss: 0.3022 | Val Acc: 0.9039 | Patience: 0/3
  >> New best: 0.3022
Epoch 04 | Train Loss: 0.1009 | Train Acc: 0.9694 | Val Loss: 0.3268 | Val Acc: 0.9041 | Patience: 0/3
Epoch 05 | Train Loss: 0.0707 | Train Acc: 0.9786 | Val Loss: 0.3466 | Val Acc: 0.9056 | Patience: 1/3
Epoch 06 | Train Loss: 0.0615 | Train Acc: 0.9804 | Val Loss: 0.3620 | Val Acc: 0.9061 | Patience: 2/3
Early stopping triggered.
S11-R2 best weights saved: Q:\scripts\projects\ComputerVision-Term9\checkpoints\s11r2_best_weights.pth


#### Best Epoch weights saved - Verification

In [34]:
resnet50.load_state_dict(torch.load(S11R2_CHECKPOINT, weights_only=True))
loss, acc, _, _ = evaluate_model(resnet50, val_loader_s11, criterion, device, 'S11-R2 verify')
print(f"Expected val loss ~0.3022 | Got: {loss:.4f}")

S11-R2 verify | Test Loss: 0.3022 | Test Acc: 0.9039 (90.39%)
Expected val loss ~0.3022 | Got: 0.3022


#### Test Set Evaluations Compare

In [35]:
resnet50.load_state_dict(torch.load(S11R2_CHECKPOINT, weights_only=True))
s11r2_loss, s11r2_acc, s11r2_preds, s11r2_labels = evaluate_model(
    resnet50, test_loader, criterion, device, 'S11-R2 -- Layer4 Only (35k)'
)

preds_np  = np.array(s11r2_preds)
labels_np = np.array(s11r2_labels)

print("\nS11-R2 Per-Class Accuracy:")
for i, name in enumerate(CLASS_NAMES):
    mask = labels_np == i
    class_acc = (preds_np[mask] == i).mean()
    print(f"  {name:<12}: {class_acc:.2%}")

S11-R2 -- Layer4 Only (35k) | Test Loss: 0.3183 | Test Acc: 0.9039 (90.39%)

S11-R2 Per-Class Accuracy:
  airplane    : 93.80%
  automobile  : 91.80%
  bird        : 86.80%
  cat         : 85.00%
  deer        : 88.10%
  dog         : 82.90%
  frog        : 92.50%
  horse       : 94.10%
  ship        : 94.10%
  truck       : 94.80%


### Layer 4 Only - using Layer 3 adapted weights

In [36]:
# S11-R3: Layer4 Only -- starting from S11-R1 (adapted layer3) checkpoint
# Hypothesis: cleaned layer3 feature maps reduce artifact noise reaching layer4
# Expected: faster convergence, better generalization than S11-R2 (layer4 from scratch)

import copy

# Load S11-R1 adapted weights as starting point
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Rebuild custom head first (required before loading state dict)
resnet50.fc = nn.Sequential(
    nn.Flatten(),
    nn.Linear(2048, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 10)
)

# Load S11-R1 checkpoint -- contains adapted layer3 + head weights
resnet50.load_state_dict(torch.load(S11R1_CHECKPOINT, weights_only=True))
resnet50 = resnet50.to(device)

# Freeze everything
for param in resnet50.parameters():
    param.requires_grad = False

# Unfreeze layer4 + head only
for param in resnet50.layer4.parameters():
    param.requires_grad = True
for param in resnet50.fc.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in resnet50.parameters() if p.requires_grad)
print(f"Trainable parameters (S11-R3): {trainable:,}")
# Expected: ~8.4M (layer4) + 271k (head) = ~8.7M

optimizer_s11r3 = optim.AdamW(
    filter(lambda p: p.requires_grad, resnet50.parameters()),
    lr=1e-4
)

print("S11-R3 ready -- layer4 on adapted layer3, AdamW lr=1e-4, 35k dataset")

# ── Training Loop ──────────────────────────────────────────────────────────────

history_s11r3 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

best_val_loss          = float('inf')
patience_counter       = 0
best_model_state_s11r3 = None

NUM_EPOCHS = 10
PATIENCE   = 3
MIN_DELTA  = 0.005

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    resnet50.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader_s11:
        images, labels = images.to(device), labels.to(device)
        optimizer_s11r3.zero_grad()
        outputs = resnet50(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_s11r3.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # --- Validate ---
    resnet50.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader_s11:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet50(images)
            loss = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total   += labels.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total

    history_s11r3['train_loss'].append(train_loss)
    history_s11r3['train_acc'].append(train_acc)
    history_s11r3['val_loss'].append(val_loss)
    history_s11r3['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"Patience: {patience_counter}/{PATIENCE}")

    # --- EarlyStopping with min_delta ---
    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state_s11r3 = copy.deepcopy(resnet50.state_dict())
        print(f"  >> New best: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

# Save checkpoint
resnet50.load_state_dict(best_model_state_s11r3)
S11R3_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 's11r3_best_weights.pth')
torch.save(best_model_state_s11r3, S11R3_CHECKPOINT)
print(f"S11-R3 best weights saved: {S11R3_CHECKPOINT}")

Trainable parameters (S11-R3): 15,235,914
S11-R3 ready -- layer4 on adapted layer3, AdamW lr=1e-4, 35k dataset
Epoch 01 | Train Loss: 0.1403 | Train Acc: 0.9566 | Val Loss: 0.2238 | Val Acc: 0.9289 | Patience: 0/3
  >> New best: 0.2238
Epoch 02 | Train Loss: 0.0839 | Train Acc: 0.9728 | Val Loss: 0.2887 | Val Acc: 0.9250 | Patience: 0/3
Epoch 03 | Train Loss: 0.0637 | Train Acc: 0.9801 | Val Loss: 0.2341 | Val Acc: 0.9324 | Patience: 1/3
Epoch 04 | Train Loss: 0.0499 | Train Acc: 0.9838 | Val Loss: 0.2495 | Val Acc: 0.9329 | Patience: 2/3
Early stopping triggered.
S11-R3 best weights saved: Q:\scripts\projects\ComputerVision-Term9\checkpoints\s11r3_best_weights.pth


#### Best Epoch weights saved - Verification

In [37]:
resnet50.load_state_dict(torch.load(S11R3_CHECKPOINT, weights_only=True))
loss, acc, _, _ = evaluate_model(resnet50, val_loader_s11, criterion, device, 'S11-R3 verify')
print(f"Expected val loss ~0.2238 | Got: {loss:.4f}")

S11-R3 verify | Test Loss: 0.2238 | Test Acc: 0.9289 (92.89%)
Expected val loss ~0.2238 | Got: 0.2238


#### Test Set Evaluations Compare

In [38]:
resnet50.load_state_dict(torch.load(S11R3_CHECKPOINT, weights_only=True))
s11r3_loss, s11r3_acc, s11r3_preds, s11r3_labels = evaluate_model(
    resnet50, test_loader, criterion, device, 'S11-R3 -- Layer4 on Layer3 (35k)'
)

preds_np  = np.array(s11r3_preds)
labels_np = np.array(s11r3_labels)

print("\nS11-R3 Per-Class Accuracy:")
for i, name in enumerate(CLASS_NAMES):
    mask = labels_np == i
    class_acc = (preds_np[mask] == i).mean()
    print(f"  {name:<12}: {class_acc:.2%}")

S11-R3 -- Layer4 on Layer3 (35k) | Test Loss: 0.2411 | Test Acc: 0.9285 (92.85%)

S11-R3 Per-Class Accuracy:
  airplane    : 95.70%
  automobile  : 96.60%
  bird        : 90.40%
  cat         : 86.00%
  deer        : 93.60%
  dog         : 87.50%
  frog        : 96.20%
  horse       : 91.30%
  ship        : 96.20%
  truck       : 95.00%
